# 面试问题：无限流式对话怎样限制 KV Cache？Sliding Window、Attention Sink 与绝对位置如何实现？

**一句话回答**：固定窗口只保留最近 token 会移除模型高度依赖的初始 attention sink，导致分布崩坏；StreamingLLM 类策略保留少量最初 token，再保留最近 rolling window，使 KV 容量固定。被保留 token 仍使用原始绝对 position，不能在搬移后从零重编号；它维持局部流式建模，不等于拥有无限语义记忆。

本 Notebook 手写 sink+recent 索引、滚动 KV 状态、受限 attention、RoPE 位置反例、完整 mask、容量估算与长程评测合同，并为关键缓存操作添加中文注释。


In [ ]:
import math
import numpy as np

# 受控序列让缓存索引和绝对位置可直接检查。
SEED143=14301; rng143=np.random.default_rng(SEED143)
assert SEED143==14301
assert np.isfinite(rng143.normal())
assert 4+12==16


## 1. 缓存策略明确 sink、recent 与总容量

对当前已见长度 `n`，保留 `[0,sink)` 和最后 `recent` 个绝对位置，重叠时去重。总物理容量最多 `sink+recent`。sink 是模型注意力行为产生的稳定锚点，不应误解为语义最重要的历史消息。


In [ ]:
def retained_positions143(n,sink,recent):
    # 绝对位置排序后去重，短序列阶段自然保留全部 token。
    return sorted(set(range(min(sink,n)))|set(range(max(0,n-recent),n)))
assert retained_positions143(5,2,4)==[0,1,2,3,4]
assert retained_positions143(20,2,4)==[0,1,16,17,18,19]
assert len(retained_positions143(100,4,12))==16


## 2. Rolling cache 存绝对 position，而非数组槽位位置

每 append 新 K/V 后按策略淘汰；数组第 3 个槽位可能对应绝对 token 98。attention、RoPE、trace 和引用都必须读取 position metadata。生产实现还需 block 引用计数与 in-flight kernel 安全释放。


In [ ]:
class RollingKV143:
    def __init__(self,sink,recent): self.sink=sink; self.recent=recent; self.items=[]
    def append(self,pos,k,v):
        # 先加入新 token，再按绝对位置策略过滤旧 KV。
        self.items.append((pos,np.array(k),np.array(v))); keep=set(retained_positions143(pos+1,self.sink,self.recent)); self.items=[x for x in self.items if x[0] in keep]
    @property
    def positions(self): return [x[0] for x in self.items]
cache143=RollingKV143(2,4)
for i in range(10): cache143.append(i,[i,1],[i,-1])
assert cache143.positions==[0,1,6,7,8,9]
assert len(cache143.items)<=6
assert cache143.items[-1][0]==9


## 3. 受限 attention 只对实际保留项归一化

query 对 sink 与 recent K 计算 score，softmax 分母不包含已淘汰或未初始化槽位。这个输出一般不等于 full attention，是部署时接受的上下文近似；必须用目标模型和真实长序列评测，而不是只测张量形状。


In [ ]:
def cached_attention143(q,cache):
    # 从逻辑缓存项堆叠 K/V，避免把空容量纳入 softmax。
    K=np.stack([x[1] for x in cache.items]); V=np.stack([x[2] for x in cache.items]); s=K@q/math.sqrt(len(q)); p=np.exp(s-s.max()); p/=p.sum(); return p@V,p
out143,p143=cached_attention143(np.array([.1,.5]),cache143)
assert out143.shape==(2,)
assert math.isclose(float(p143.sum()),1.0)
assert len(p143)==len(cache143.items)


## 4. 合成反例展示为什么只留最近窗口可能丢锚点

构造一个 query 对位置 0 分数最高的例子：recent-only 淘汰它后输出显著改变，而 sink+recent 保留。真实论文发现的是模型统计现象，不代表任意任务都应固定保留相同数量；sink 数要通过 perplexity 与任务实验选。


In [ ]:
scores143=np.array([12.,0.,0.,0.,0.,0.]); values143=np.array([10.,1.,1.,1.,1.,1.])
def weighted143(scores,values,indices):
    # 在选中集合内重新归一化，模拟 cache 淘汰后的注意力。
    s=scores[indices]; p=np.exp(s-s.max()); p/=p.sum(); return float(p@values[indices])
full_value143=weighted143(scores143,values143,list(range(6))); recent_value143=weighted143(scores143,values143,[2,3,4,5]); sink_value143=weighted143(scores143,values143,[0,2,3,4,5])
assert abs(sink_value143-full_value143)<.01
assert abs(recent_value143-full_value143)>5
assert recent_value143==1.0


## 5. RoPE 使用原始绝对位置，淘汰后不能重置

RoPE 旋转角取决于 token position。若把保留的 `[0,1,98,99]` 重编号成 `[0,1,2,3]`，Q/K 相位关系变化，cache 与模型已生成的状态不一致。下面用二维旋转直接证明位置重置改变向量。


In [ ]:
def rotate143(x,pos,theta=.1):
    # 二维 RoPE 示例保留绝对位置产生的旋转相位。
    c,s=math.cos(pos*theta),math.sin(pos*theta); return np.array([c*x[0]-s*x[1],s*x[0]+c*x[1]])
vec143=np.array([1.,0.]); abs143=rotate143(vec143,98); reset143=rotate143(vec143,2)
assert not np.allclose(abs143,reset143)
assert math.isclose(np.linalg.norm(abs143),1.0)
assert np.allclose(rotate143(vec143,98),abs143)


## 6. 训练/全序列 oracle 可显式构造 sink-window mask

第 i 个 query 可看初始 sink 区，以及最近 window 内且不晚于 i 的 keys。mask 用于验证 rolling kernel，但全 `T×T` 版本不用于真正无限流式部署。padding 与 segment 边界还要在此基础上相交。


In [ ]:
def streaming_mask143(n,sink,recent):
    # 每行保留 sink 和包含当前位置的最近 recent 个绝对 key。
    M=np.zeros((n,n),dtype=bool)
    for i in range(n): M[i,retained_positions143(i+1,sink,recent)]=True
    return M
M143=streaming_mask143(10,2,4)
assert M143[9].nonzero()[0].tolist()==[0,1,6,7,8,9]
assert not M143[3,9]
assert np.all(np.diag(M143))


## 7. 固定容量不等于固定端到端成本

KV 元素最多 `sink+recent`，所以 decode attention 每步读取固定历史规模；但总生成时间仍随输出 token 数线性增长，且长会话还需要外部 memory/检索、摘要和工具状态。容量估算乘层数、KV heads、head dim、K/V 两份与 dtype。


In [ ]:
def cache_bytes143(layers,hkv,d,sink,recent,bytes_=2):
    # Key 和 Value 各一份，因此乘 2。
    return 2*layers*hkv*d*(sink+recent)*bytes_
fixed143=cache_bytes143(32,8,128,4,4092); full143=cache_bytes143(32,8,128,4,999_996)
assert full143>100*fixed143
assert cache_bytes143(32,1,128,4,4092)==fixed143/8
assert fixed143>0


## 8. 评测稳定建模、任务记忆和系统性能三个层次

在超训练长度语料上画 position→NLL/perplexity，检查是否出现断崖；在 long-horizon QA/Agent 任务测早期事实 recall，明确 sink 不保存语义；系统侧测固定 HBM、TPOT、eviction 和 cache invariant。与 full、recent-only、sink+recent、摘要/RAG 做消融。


In [ ]:
nll_recent143=np.array([2.1,2.2,3.0,7.5]); nll_sink143=np.array([2.1,2.2,2.3,2.4])
# 尾段 NLL 斜率作为是否发生流式崩坏的简化探针。
jump_recent143=float(np.max(np.diff(nll_recent143))); jump_sink143=float(np.max(np.diff(nll_sink143)))
assert jump_recent143>jump_sink143
assert len(cache143.items)==len(set(cache143.positions))
assert cache143.positions==sorted(cache143.positions)


## 面试总结

完整回答是：**定义 sink+recent 固定容量 → cache item 保存绝对 position → append 后按逻辑策略淘汰 → softmax 只含真实保留 K/V → 合成与真实实验比较 recent-only → RoPE 不重编号 → 全矩阵 mask 做 oracle → 容量乘 K/V、层、Hkv、D、dtype → position-NLL、长程 recall 与 TPOT 联合评测**。StreamingLLM 稳定局部流式 attention，不是无限记忆系统。

延伸阅读：[StreamingLLM](https://arxiv.org/abs/2309.17453)、[Longformer](https://arxiv.org/abs/2004.05150)、[Mistral 7B](https://arxiv.org/abs/2310.06825)。
